In [2]:
import pandas as pd 

flows = pd.DataFrame({
    "src_ip": ["192.168.56.11", "192.168.56.11", "10.0.0.5", "10.0.0.7"],
    "dst_port": [80, 22, 443, 22],
    "packets": [12, 950, 30, 4],
    "duration_s": [1.2, 0.8, 3.5, 0.1],
    "label": ["normal", "attack", "normal", "attack"],
})
flows

,src_ip,dst_port,packets,duration_s,label
0,192.168.56.11,80,12,1.2,normal
1,192.168.56.11,22,950,0.8,attack
2,10.0.0.5,443,30,3.5,normal
3,10.0.0.7,22,4,0.1,attack


In [3]:
print(flows.shape)
print(flows["label"].value_counts())
flows.describe()

(4, 5)
label
normal    2
attack    2
Name: count, dtype: int64


,dst_port,packets,duration_s
count,4.000000,4.000000,4.00000
mean,141.750000,249.000000,1.40000
std,202.685923,467.459802,1.47196
min,22.000000,4.000000,0.10000
25%,22.000000,10.000000,0.62500
50%,51.000000,21.000000,1.00000
75%,170.750000,260.000000,1.77500
max,443.000000,950.000000,3.50000


In [4]:
df = pd.read_csv("../data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
print(df.shape)

(286467, 79)


In [5]:
print(df.columns.tolist())

[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count', ' URG Flag 

In [6]:
df.columns = df.columns.str.strip()
print(df["Label"].value_counts())

Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64


In [7]:
df.groupby("Label")[["Flow Duration", "Total Fwd Packets", "Destination Port"]].median()

,Flow Duration,Total Fwd Packets,Destination Port
Label,,,
BENIGN,30964.0,2.0,80.0
PortScan,47.0,1.0,3690.0


In [8]:
print(df.groupby("Label")["Destination Port"].nunique())
print(df[df["Label"] == "BENIGN"]["Destination Port"].value_counts().head(5))

Label
BENIGN      13694
PortScan     1000
Name: Destination Port, dtype: int64
Destination Port
53     58585
443    26695
80     16368
22       672
123      621
Name: count, dtype: int64


In [9]:
import numpy as np
print(df.isna().sum().sum())
print(np.isinf(df.select_dtypes(include="number")).sum().sum())

15
727


In [10]:
bad = df.replace([np.inf, -np.inf], np.nan).isna().sum()
print(bad[bad > 0])

Flow Bytes/s      371
Flow Packets/s    371
dtype: int64


In [11]:
df = df.replace([np.inf, -np.inf], np.nan).dropna()
print(df.shape)
print(df["Label"].value_counts())

(286096, 79)
Label
PortScan    158804
BENIGN      127292
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split

y = (df["Label"] == "PortScan").astype(int)
X = df.drop(columns=["Label"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(228876, 78) (57220, 78)
0.5550734895751411 0.5550681579867179


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["BENIGN", "PortScan"]))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     25459
    PortScan       1.00      1.00      1.00     31761

    accuracy                           1.00     57220
   macro avg       1.00      1.00      1.00     57220
weighted avg       1.00      1.00      1.00     57220

[[25458     1]
 [    4 31757]]


In [14]:
for i, name in enumerate(X_train.columns):
    print(i, name)

0 Destination Port
1 Flow Duration
2 Total Fwd Packets
3 Total Backward Packets
4 Total Length of Fwd Packets
5 Total Length of Bwd Packets
6 Fwd Packet Length Max
7 Fwd Packet Length Min
8 Fwd Packet Length Mean
9 Fwd Packet Length Std
10 Bwd Packet Length Max
11 Bwd Packet Length Min
12 Bwd Packet Length Mean
13 Bwd Packet Length Std
14 Flow Bytes/s
15 Flow Packets/s
16 Flow IAT Mean
17 Flow IAT Std
18 Flow IAT Max
19 Flow IAT Min
20 Fwd IAT Total
21 Fwd IAT Mean
22 Fwd IAT Std
23 Fwd IAT Max
24 Fwd IAT Min
25 Bwd IAT Total
26 Bwd IAT Mean
27 Bwd IAT Std
28 Bwd IAT Max
29 Bwd IAT Min
30 Fwd PSH Flags
31 Bwd PSH Flags
32 Fwd URG Flags
33 Bwd URG Flags
34 Fwd Header Length
35 Bwd Header Length
36 Fwd Packets/s
37 Bwd Packets/s
38 Min Packet Length
39 Max Packet Length
40 Packet Length Mean
41 Packet Length Std
42 Packet Length Variance
43 FIN Flag Count
44 SYN Flag Count
45 RST Flag Count
46 PSH Flag Count
47 ACK Flag Count
48 URG Flag Count
49 CWE Flag Count
50 ECE Flag Count
51 Down/

In [15]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(10))

Fwd Packet Length Mean         0.066291
Flow IAT Max                   0.066284
Bwd Packets/s                  0.065682
Average Packet Size            0.064058
Subflow Fwd Bytes              0.063421
Total Length of Fwd Packets    0.058454
Fwd Packet Length Max          0.055073
Avg Fwd Segment Size           0.047518
Flow Duration                  0.046930
PSH Flag Count                 0.043320
dtype: float64


In [16]:
X_train2 = X_train.drop(columns=["Destination Port"])
X_test2 = X_test.drop(columns=["Destination Port"])

model2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model2.fit(X_train2, y_train)
print(confusion_matrix(y_test, model2.predict(X_test2)))

[[25458     1]
 [    4 31757]]


In [17]:
print(X_train2.shape)

(228876, 77)


In [18]:
from sklearn.ensemble import IsolationForest

X_benign = X_train[y_train == 0]

iso = IsolationForest(n_estimators=100, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(X_benign)

pred = iso.predict(X_test)
y_pred_iso = (pred == -1).astype(int)

print(confusion_matrix(y_test, y_pred_iso))
print(classification_report(y_test, y_pred_iso, target_names=["BENIGN", "PortScan"]))

[[25186   273]
 [31761     0]]
              precision    recall  f1-score   support

      BENIGN       0.44      0.99      0.61     25459
    PortScan       0.00      0.00      0.00     31761

    accuracy                           0.44     57220
   macro avg       0.22      0.49      0.31     57220
weighted avg       0.20      0.44      0.27     57220



In [19]:
from sklearn.metrics import roc_auc_score

scores = -iso.score_samples(X_test)
print(roc_auc_score(y_test, scores))
print(pd.Series(scores, index=y_test.index).groupby(y_test).median())

0.5592807382300824
Label
0    0.347503
1    0.352176
dtype: float64


In [20]:
ddos = pd.read_csv("../data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
ddos.columns = ddos.columns.str.strip()
ddos = ddos.replace([np.inf, -np.inf], np.nan).dropna()
print(ddos.shape)
print(ddos["Label"].value_counts())

(225711, 79)
Label
DDoS      128025
BENIGN     97686
Name: count, dtype: int64


In [21]:
X_ddos = ddos.drop(columns=["Label"])
y_ddos = (ddos["Label"] == "DDoS").astype(int)

pred_ddos = model.predict(X_ddos)
pd.crosstab(y_ddos, pred_ddos, rownames=["Real"], colnames=["Guess"])

Guess,0,1
Real,,
0,97686,0
1,53857,74168


In [22]:
pred_iso_ddos = iso.predict(X_ddos)
y_pred_iso_ddos = (pred_iso_ddos == -1).astype(int)
pd.crosstab(y_ddos, y_pred_iso_ddos, rownames=["Real"], colnames=["Guess"])

Guess,0,1
Real,,
0,88359,9327
1,106596,21429


## Findings

**Port scans.** The Random Forest made 5 mistakes out of 57,220 test connections, the 20% it never saw during training. Scans were easy to spot because of how they behave, not which port they hit: a scan connection lasts about 47 microseconds and sends about 1 packet, while a normal one lasts about 30,964 microseconds. Removing the destination port changed nothing.

**DDoS, an attack the models never studied.** The Random Forest caught 74,168 of 128,025 attacks (58%) with 0 false alarms. The Isolation Forest caught only 21,429 (17%) and wrongly flagged 9,327 normal connections (9.5%), even though it was set to flag about 1%.

**Why the Isolation Forest failed on port scans.** It flags connections that look unusual next to normal traffic. A single scan connection is tiny and fast, and normal traffic probably contains many connections like that, so on its own it looks normal. A scan only shows as a pattern across many connections, and the model reads one at a time. It caught 0 of 31,761 scans.

**Realistic traffic (attacks at 1%).** On the DDoS file with attacks reduced to 1% of connections, the Random Forest raised 555 alerts, all real, but missed 432 attacks. The Isolation Forest raised 9,498 alerts and only 171 were real (about 2%), so an analyst would be flooded with false alarms.

**Training on all three attack families.** A Random Forest trained on port scan, DDoS and brute force (FTP and SSH), and tested on held-out connections, made 14 mistakes out of 191,491 with no false alarms. It also names the attack type and never confused one attack for another. Before this, a model trained on port scans alone caught 1 of 13,832 brute force attacks, and adding DDoS to its training did not help (0 caught). The detector only recognizes attack families it has been trained on. These scores are probably optimistic, because the test connections come from the same attack runs as the training ones.

**Limits.**
- 55% of this file is attacks. On a real network attacks are a tiny fraction, so these scores will not carry over.
- The models studied one day and one attack type. On a different attack the Random Forest missed 42%.
- Still to test: mostly normal traffic, and my own lab traffic.

In [23]:
normal_idx = y_ddos[y_ddos == 0].index
attack_idx = y_ddos[y_ddos == 1].sample(n=987, random_state=42).index
idx = normal_idx.union(attack_idx)

X_r = X_ddos.loc[idx]
y_r = y_ddos.loc[idx]
print(y_r.shape, y_r.mean())

rf_pred = model.predict(X_r)
iso_pred = (iso.predict(X_r) == -1).astype(int)

print("Random Forest")
print(confusion_matrix(y_r, rf_pred))
print("Isolation Forest")
print(confusion_matrix(y_r, iso_pred))

(98673,) 0.010002736310844912
Random Forest
[[97686     0]
 [  432   555]]
Isolation Forest
[[88359  9327]
 [  816   171]]


In [24]:
tue = pd.read_csv("../data/Tuesday-WorkingHours.pcap_ISCX.csv")
tue.columns = tue.columns.str.strip()
tue = tue.replace([np.inf, -np.inf], np.nan).dropna()
print(tue.shape)
print(tue["Label"].value_counts())

(445645, 79)
Label
BENIGN         431813
FTP-Patator      7935
SSH-Patator      5897
Name: count, dtype: int64


In [25]:
X_tue = tue.drop(columns=["Label"])

rf_pred = model.predict(X_tue)
iso_pred = (iso.predict(X_tue) == -1).astype(int)

print("Random Forest")
print(pd.crosstab(tue["Label"], rf_pred, rownames=["Real"], colnames=["Guess"]))
print("Isolation Forest")
print(pd.crosstab(tue["Label"], iso_pred, rownames=["Real"], colnames=["Guess"]))

Random Forest
Guess             0   1
Real                   
BENIGN       431797  16
FTP-Patator    7935   0
SSH-Patator    5896   1
Isolation Forest
Guess             0     1
Real                     
BENIGN       427784  4029
FTP-Patator    7935     0
SSH-Patator    5897     0


In [26]:
combo = pd.concat([df, ddos], ignore_index=True)
X_combo = combo.drop(columns=["Label"])
y_combo = (combo["Label"] != "BENIGN").astype(int)

rf2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf2.fit(X_combo, y_combo)

rf2_pred = rf2.predict(X_tue)
pd.crosstab(tue["Label"], rf2_pred, rownames=["Real"], colnames=["Guess"])

Guess,0
Real,
BENIGN,431813
FTP-Patator,7935
SSH-Patator,5897


In [27]:
all_df = pd.concat([df, ddos, tue], ignore_index=True)
X_all = all_df.drop(columns=["Label"])
y_all = all_df["Label"]

Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

rf3 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf3.fit(Xa_train, ya_train)

rf3_pred = rf3.predict(Xa_test)
pd.crosstab(ya_test, rf3_pred, rownames=["Real"], colnames=["Guess"])

Guess,BENIGN,DDoS,FTP-Patator,PortScan,SSH-Patator
Real,,,,,
BENIGN,131359,0,0,0,0
DDoS,8,25597,0,0,0
FTP-Patator,2,0,1585,0,0
PortScan,2,0,0,31759,0
SSH-Patator,2,0,0,0,1177
